
# Modeling example




**Running in Google Colab?** Run the next cell first. It installs GemPy (takes 1-2 minutes).
If Colab asks you to restart the session afterwards, click *Restart* and continue with the cell after it.

Running locally (or on Binder)? The cell does nothing, so you can run it or skip it.

In [ ]:
# Install GemPy - only needed in Google Colab
import sys
if "google.colab" in sys.modules:
    %pip install -q "gempy[opt]==2026.0.3" "gempy_engine==2026.0.3.post1" "gempy_viewer[opt]==2026.0.5" "gempy_plugins==2026.0.3" "pyvista==0.49.2" "trame-pyvista==0.3.6" "nest_asyncio2==1.7.4" "anywidget==0.11.0"
else:
    print("Not running in Colab - nothing to install.")

In [ ]:
# Required imports
import gempy as gp
import gempy_viewer as gpv
import numpy as np

In [ ]:
# 3D plot settings for Colab: show 3D plots directly in the notebook
# (locally nothing changes - you can skip this cell)
import sys
if "google.colab" in sys.modules:
    import gempy_viewer as gpv

    PLOT_3D_BACKEND = 'html'  # interactive; use 'static' for plain images if 'html' doesn't work

    # gpv.plot_3d normally opens a separate window, which Colab can't show.
    # This wrapper makes it draw inside the notebook instead.
    _plot_3d = gpv.plot_3d
    def _plot_3d_colab(*args, show=True, image=False, kwargs_plotter=None, **kwargs):
        if image:
            return _plot_3d(*args, show=show, image=image, kwargs_plotter=kwargs_plotter, **kwargs)
        kwargs_plotter = {'notebook': True, **(kwargs_plotter or {})}
        gempy_vista = _plot_3d(*args, show=False, kwargs_plotter=kwargs_plotter, **kwargs)
        if show:
            gempy_vista.p.show(jupyter_backend=PLOT_3D_BACKEND)
        return gempy_vista
    gpv.plot_3d = _plot_3d_colab

### 1) Setting up a model

In [ ]:
# Define the path to data
data_path = 'https://raw.githubusercontent.com/cgre-aachen/gempy_data/master/'
path_to_data = data_path + "/data/input_data/jan_models/"

In [ ]:
# Create a GeoModel instance
geo_model = gp.create_geomodel(
    project_name='Workshop_example',
    extent=[0, 2500, 0, 1000, 0, 1000],
    resolution=[125, 50, 50],
    importer_helper=gp.data.ImporterHelper(
        path_to_orientations=path_to_data + "model7_orientations.csv",
        path_to_surface_points=path_to_data + "model7_surface_points.csv"
    )
)

In [ ]:
# Display a basic cross section of input data
gpv.plot_2d(geo_model)

### 2) The Structural frame

In [ ]:
# Print the strucutral frame
geo_model.structural_frame

In [ ]:
# Map geological series to surfaces
gp.map_stack_to_surfaces(
    gempy_model=geo_model,
    mapping_object={
        "Fault_Series" : ('fault'),
        "Strat_Series1": ('rock3'),
        "Strat_Series2": ('rock2', 'rock1'),
    }
)

In [ ]:
# Define youngest structural group as fault
gp.set_is_fault(geo_model, ["Fault_Series"])

### 3) Computing a solution

In [ ]:
# Compute a solution for the model
gp.compute_model(geo_model)

In [ ]:
# View interpolation options
geo_model.interpolation_options

In [ ]:
# Some examples of how to access results
print(geo_model.solutions.raw_arrays.lith_block)
print(geo_model.grid.dense_grid.values)

### 4) Visualization

In [ ]:
# Display the result in 2d section
gpv.plot_2d(geo_model, cell_number=25)

In [ ]:
# Display the scalar field of the fault in 2d section
gpv.plot_2d(geo_model, show_scalar=True, show_lith=False, series_n=0)
gpv.plot_2d(geo_model, show_scalar=True, show_lith=False, series_n=1)
gpv.plot_2d(geo_model, show_scalar=True, show_lith=False, series_n=2)

In [ ]:
# Display the result in 3D
gpv.plot_3d(geo_model)

### 5) Adding topography

In [ ]:
# Setting a randomly generated topography
gp.set_topography_from_random(
    grid=geo_model.grid,
    fractal_dimension=2,
    d_z=np.array([700, 950]),
    topography_resolution=np.array([125, 50])
)

In [ ]:
# Recompute model as a new grid was added
gp.compute_model(geo_model)

In [ ]:
# Display a cross-section with topography
gpv.plot_2d(geo_model, show_topography=True)

In [ ]:
# Displaying a geological map
gpv.plot_2d(geo_model, show_topography=True, section_names=['topography'], show_boundaries=False, show_data=False)

In [ ]:
# Display the 3d model with topography
gpv.plot_3d(geo_model, show_lith=True, show_topography=True)